# Monday, April 27th, 2026

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Conway's Game of Life

Conway's [Game of Life](https://conwaylife.com/wiki/Conway%27s_Game_of_Life) is a cellular automaton created by John Conway in 1970. It is a deterministic process where the next state of a population of cells depends only on the current state. We will use 2D NumPy arrays to represent the population of cells aranged in an $n \times n$ grid.
A value of `1` will signify that a cell is alive while a value of `0` will signify that a cell is dead.

### Starting configuration

Lets begin with an $ n\times n $ array of all 0s with a small three-block column (3$\times$1) of 1s in the middle. Use an integer datatype (`dtype=int`) when defining your array.

**Exercise:**  Write a function `starting_state(n)` that returns the array described above.

### Rules of Life

We will use the current state of the population to determine the next state.
In the Game of Life, each cell interacts with its eight neighbors (i.e. the horizontally, vertically, or diagonally adjacent cells).

![neighbors](https://jllottes.github.io/_images/epidemic-2.svg)

The rules of the Game of Life can be summarized as follows:

 1. Any live cell with two or three live neighbors survives.
 2. Any dead cell with with three live neighbors becomes a live cell.
 3. All other live cells die in the next generation, and all other dead cells stay dead.

### Counting the number of live neighbors

In order to update our array from one state to the next, we need to be able to count the number of live neighbors of the $(i,j)$th cell for any choice of $i,j$.

**Exercise:** Write a function `count_live_neighbors(cells,i,j)` that counts the number of living neighbors of the $(i,j)$th cell.

 - We handled a similar problem with the [Image Denoising](https://jllottes.github.io/Projects/image_denoising/image_denoising) project.
 - How can we handle cells on the edge of the grid?
 - The `np.sum` function will add all values in an array.
 - We want to exclude (or remove from the sum) the $(i,j)$th cell when counting the number of living neighbors.



Let's test this on our configuration of a `3` by `1` column of live cells:

### Updating the `cells` population

We can now update the `cells` array according to the rules.  We have to update every entry of the array, so we will need to loop through all the entries.

**Exercise:** Write a function `update_cells(cells)` that takes in a population array `cells`, applies the Rules of Life to update the population, and returns the updated population.

### Animating the dynamics

All of the figures that we've generated so far have been static, i.e. they do not change. Python also allows us to create interactive figures that can change over time. 

An easy way to activate this mode is to use the "magic", `%matplotlib qt`. Magics like this are not actual Python code, but instead are signals to Jupyter notebook that can be used to change various configuration options. This particular magic will enable interactive plotting and switch to using external plotting windows, instead of the inline plotting (i.e. appearing within the notebook) that we've been using.

*Note: Once we are finished with any interactive plotting, we can return to inline plotting using the magic `%matplotlib inline`.*

In [ ]:
%matplotlib qt

Once we are in interactive mode, the `FuncAnimation` function from `matplotlib.animation` can be used to create animations.
It takes in a figure `fig` and function `animate`. The `animate` function should take in a frame index `i` and perform any desired updates to the figure.

For example, if we've stored the output of `plt.imshow` as `im`, we can update the displayed array using the `.set_data` method. That is, `im.set_data(new_array)` will change the figure to display the colorized data from `new_array`.

In [ ]:
from matplotlib.animation import FuncAnimation

**Exercise:** Modify the code below to animate the Game of Life.

In [ ]:
x = np.zeros((200,200), dtype=int)

fig = plt.figure()
im = plt.imshow(x,vmin=0,vmax=1)

def animate(i):
    x[:,:] = np.random.random(x.shape) < .05
    im.set_data(x)
    return im

anim = FuncAnimation(fig,animate,save_count=100)
plt.show()

**Exercise:** Use `np.random.random` to randomly select an initial `cells` array of `0`s and `1`s.

Let's create a function `animate_game_of_life` that takes in an array `cells` specifying the initial configuration and animates the Game of Life.

*Note: You may need this function to return the animation object returned by `FuncAnimation` (and then store the returned animation object when calling the function) in order for the animation to run properly.*

What can we do to make our code run more efficiently? At a `200` by `200` grid, it is very slow to update. Let's look at the `count_live_neighbors` function, which will run $n^2$ times for an $n$ by $n$ grid. Any improvements we can make here will give big improvements to our code.

For example, our current `count_live_neighbors` function creates a padded version of the array before finding the neighbors. On the other hand, the padded version of the array is the same for every single cell. It would be better if we moved this operation outside of the `count_live_neighbors` function so that we only have to perform it once, then passed in the `padded_cells` array as the new input for `count_live_neighbors`.

**Exercise** Rewrite the `count_live_neighbors` to take `padded_cells` in as an input.

**Exercise**: Rewrite the `update_cells` function to generate the `padded_cells` array and pass it into the  `count_live_neighbors` function.

Let's see how this helps the speed of our animation.

## Analyzing the dynamics of the Game of Life

Letting the Game of Life play out and observing the animation helps give us a sense of the resulting dynamics, but it is difficult to make meaningful quantitative statements about what is happening. We would like be able to distill this animation into some data that we can more easily describe.

With our current code, we're constantly overwriting the `cells` array to store the current state only. It would be useful to maintain a history of all states of `cells`. Let's create a new variable, `cells_history` that will contain each iteration of `cells`. Let's also set a specific number of time steps that we will carry out.

Now, we want to iterate through the time steps and update our `cells` array and place its updated contents into the appropriate slice of `cells_history`.

Let's use the code above to create a function that simulates the Game of Life with a given initial configuration for a given number of time steps and returns an array containing the entire history of configurations.

We have all of this data, and now we want to visualize it. First, let's switch back to inline plotting using the magic, `%matplotlib inline`.

In [ ]:
%matplotlib inline

One challenge we have is with the dimensionality of our data. Each data point corresponds to a row position, a column position, and a time position.


One way to reduce the dimensionality is to look at particular time indices. For example, we could create a 2x2 grid of subplots that show different time indices.

*Note: the `enumerate` function will zip together a list of indices (starting at `0`) with a supplied iterable (e.g. list). This is useful whenever we want to iterate through a list while also having access to each element's corresponding index. In this case, we can use the index to select a subplot.*

Another strategy to reduce the dimesionality is to condense the geometric information (i.e. the information in the first two dimensions of `cells_history`) into a single number per time step so that we could plot this quantity against time (i.e. against the third dimension).

For example, we could count the number of live cells during each time step and plot this against time.

*Recall: We can use `np.sum` together with the `axis` keyword argument to compute sums through a particular dimension (or dimensions). In this case, we would want to take the sum through the row and column dimensions (but not the time dimension) to count up the total number of live cells at each time step.*

### Further analysis of the number of live cells

There are many things we can do to continue studying the dynamics of the Game of Life by looking at how the number of live cells changes over time. Here are just a few ideas.

**Exercise:** For a given choice of parameters `n` (the square grid size), `percent_live_cells` (the portion of randomly selected cells that will be initially set to living), and `T` (number of time steps to simulate), simulate the Game of Life many times. For each of these simulations, plot the number of live cells as a function of time in a single plot. Also include the average (across the numerous simulations) number of live cells per time step.

**Exercise:** For a given choice of parameters `n` and `T`, simulate the Game of Life with several different choices of `percent_live_cells` used to generate the random initial configuration. Plot the number of live cells as a function of time for these numerous simulations in a single plot.

**Exercise:** For a given choice of parameters `percent_live_cells` and `T`, simulate the Game of Life with several different choices of square grid size `n`. Instead of plotting the number of live cells as a function of time, normalize this count by the size of the grid (e.g. by dividing by `n**2`). Plot this normalized count for each of the simulations in a single plot.

**Exercise:** For a given choice of parameter `T`, simulate the Game of Life with several combinations of choices for `n` and `percent_live_cells`. For each simulation, calculate the average number of live cells from the last `10` times steps (choose `T > 10` at least). Create an array to store this data and use `plt.imshow` to visualize this array.